# LSTM Predictive Maintenance Model Training

**Operational Technology Digital Twin (OTDT) - IBM Research Lab Africa**

This notebook trains an LSTM model to predict equipment failures 30 days in advance using synthetic sensor data from the GDC geothermal plant simulation.

**DISCLAIMER**: This model uses synthetic data generated for demonstration purposes. The dataset simulates realistic sensor patterns but does not represent actual equipment behavior.

**Target Performance**: AUC-ROC > 0.82 on test set

**Runtime**: Python 3.10 with GPU (Watson Studio eu-de region)

In [ ]:
# Install required packages
!pip install tensorflow==2.15.0 pandas==2.1.4 openpyxl==3.1.2 scikit-learn==1.3.2 matplotlib==3.8.2 seaborn==0.13.0

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# Load dataset
# NOTE: Upload Sensor_Readings.xlsx to your Watson Studio project first
# Then update the path below to match your project structure
# Example: df = pd.read_excel('/project_data/data_asset/Sensor_Readings.xlsx')

df = pd.read_excel('Sensor_Readings.xlsx')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())

# Check class distribution
print(f"\n{'='*60}")
print("CLASS DISTRIBUTION")
print(f"{'='*60}")
failure_counts = df['failure_label'].value_counts()
print(f"Normal (0): {failure_counts[0]:,} ({failure_counts[0]/len(df)*100:.2f}%)")
print(f"Pre-failure (1): {failure_counts[1]:,} ({failure_counts[1]/len(df)*100:.2f}%)")
print(f"\nClass imbalance ratio: {failure_counts[0]/failure_counts[1]:.2f}:1")

# Check data types and missing values
print(f"\n{'='*60}")
print("DATA QUALITY")
print(f"{'='*60}")
print(df.info())
print(f"\nMissing values:\n{df.isnull().sum()}")

In [ ]:
# Feature Engineering
print("Starting feature engineering...")

# Sort by asset and timestamp
df = df.sort_values(['asset_id', 'timestamp']).reset_index(drop=True)

# Define sensor columns
sensor_cols = ['temperature_c', 'pressure_bar', 'vibration_mm_s', 
               'flow_rate_kg_s', 'rotation_rpm']

# Initialize feature dataframe
feature_df = df.copy()

# 1. Rolling statistics (7-day window = 168 hours)
print("Computing rolling statistics...")
window_size = 168
for col in sensor_cols:
    feature_df[f'{col}_rolling_mean'] = feature_df.groupby('asset_id')[col].transform(
        lambda x: x.rolling(window=window_size, min_periods=1).mean()
    )
    feature_df[f'{col}_rolling_std'] = feature_df.groupby('asset_id')[col].transform(
        lambda x: x.rolling(window=window_size, min_periods=1).std()
    ).fillna(0)

# 2. Rate of change (first derivative)
print("Computing rate of change...")
for col in sensor_cols:
    feature_df[f'{col}_rate_of_change'] = feature_df.groupby('asset_id')[col].transform(
        lambda x: x.diff()
    ).fillna(0)

# 3. Days since last maintenance (derived from health_score trend)
print("Computing days since maintenance...")
# Detect maintenance events (health_score increases significantly)
feature_df['health_score_diff'] = feature_df.groupby('asset_id')['health_score'].diff().fillna(0)
feature_df['maintenance_event'] = (feature_df['health_score_diff'] > 10).astype(int)

# Calculate hours since last maintenance
def calc_hours_since_maintenance(group):
    hours_since = []
    counter = 0
    for maint in group['maintenance_event']:
        if maint == 1:
            counter = 0
        else:
            counter += 1
        hours_since.append(counter)
    return pd.Series(hours_since, index=group.index)

feature_df['hours_since_maintenance'] = feature_df.groupby('asset_id').apply(
    calc_hours_since_maintenance
).reset_index(level=0, drop=True)

feature_df['days_since_maintenance'] = feature_df['hours_since_maintenance'] / 24

# 4. Add health_score as a feature
feature_df['health_score_normalized'] = feature_df['health_score']

# Select final feature columns
feature_columns = (
    sensor_cols + 
    [f'{col}_rolling_mean' for col in sensor_cols] +
    [f'{col}_rolling_std' for col in sensor_cols] +
    [f'{col}_rate_of_change' for col in sensor_cols] +
    ['days_since_maintenance', 'health_score_normalized']
)

print(f"\nTotal features created: {len(feature_columns)}")
print(f"Feature list: {feature_columns}")

# 5. Normalize features
print("\nNormalizing features...")
scaler = StandardScaler()
feature_df[feature_columns] = scaler.fit_transform(feature_df[feature_columns])

print("Feature engineering complete!")
print(f"\nFeature statistics:")
print(feature_df[feature_columns].describe())

In [ ]:
# Build sequences
print("Building sequences...")

SEQUENCE_LENGTH = 720  # 30 days at hourly resolution
STEP_SIZE = 1  # Sliding window step

def create_sequences(data, feature_cols, target_col, seq_length, step_size):
    """
    Create sequences for LSTM training.
    
    Args:
        data: DataFrame with features and target
        feature_cols: List of feature column names
        target_col: Target column name
        seq_length: Length of each sequence
        step_size: Step size for sliding window
    
    Returns:
        X: Array of shape (samples, seq_length, n_features)
        y: Array of shape (samples,)
        asset_ids: List of asset IDs for each sequence
    """
    X_list = []
    y_list = []
    asset_list = []
    
    # Process each asset separately
    for asset_id in data['asset_id'].unique():
        asset_data = data[data['asset_id'] == asset_id].reset_index(drop=True)
        
        # Create sequences with sliding window
        for i in range(0, len(asset_data) - seq_length, step_size):
            # Extract sequence
            sequence = asset_data.iloc[i:i+seq_length][feature_cols].values
            # Target is the label at the end of the sequence
            target = asset_data.iloc[i+seq_length-1][target_col]
            
            X_list.append(sequence)
            y_list.append(target)
            asset_list.append(asset_id)
    
    X = np.array(X_list)
    y = np.array(y_list)
    
    return X, y, asset_list

# Create sequences
X, y, asset_ids = create_sequences(
    feature_df, 
    feature_columns, 
    'failure_label',
    SEQUENCE_LENGTH, 
    STEP_SIZE
)

print(f"\nSequence creation complete!")
print(f"X shape: {X.shape} (samples, timesteps, features)")
print(f"y shape: {y.shape}")
print(f"\nClass distribution in sequences:")
unique, counts = np.unique(y, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  Class {int(label)}: {count:,} ({count/len(y)*100:.2f}%)")

In [ ]:
# Train/Validation/Test Split (60/20/20 by time)
print("Splitting data by time (no shuffle for time series)...")

n_samples = len(X)
train_size = int(0.6 * n_samples)
val_size = int(0.2 * n_samples)

# Split indices
train_end = train_size
val_end = train_size + val_size

# Create splits
X_train = X[:train_end]
y_train = y[:train_end]

X_val = X[train_end:val_end]
y_val = y[train_end:val_end]

X_test = X[val_end:]
y_test = y[val_end:]

print(f"\nSplit sizes:")
print(f"  Train: {X_train.shape[0]:,} samples ({X_train.shape[0]/n_samples*100:.1f}%)")
print(f"  Val:   {X_val.shape[0]:,} samples ({X_val.shape[0]/n_samples*100:.1f}%)")
print(f"  Test:  {X_test.shape[0]:,} samples ({X_test.shape[0]/n_samples*100:.1f}%)")

print(f"\nClass distribution per split:")
for split_name, split_y in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    unique, counts = np.unique(split_y, return_counts=True)
    print(f"\n  {split_name}:")
    for label, count in zip(unique, counts):
        print(f"    Class {int(label)}: {count:,} ({count/len(split_y)*100:.2f}%)")

In [ ]:
# Build LSTM Model
print("Building LSTM model...")

n_features = X_train.shape[2]
sequence_length = X_train.shape[1]

model = Sequential([
    # Input layer
    LSTM(64, return_sequences=True, input_shape=(sequence_length, n_features), name='lstm_1'),
    Dropout(0.2, name='dropout_1'),
    
    # Second LSTM layer
    LSTM(64, return_sequences=True, name='lstm_2'),
    Dropout(0.2, name='dropout_2'),
    
    # Third LSTM layer
    LSTM(64, return_sequences=False, name='lstm_3'),
    Dropout(0.2, name='dropout_3'),
    
    # Dense layer
    Dense(32, activation='relu', name='dense_1'),
    
    # Output layer
    Dense(1, activation='sigmoid', name='output')
])

# Compile model
optimizer = Adam(learning_rate=0.001)
model.compile(
    optimizer=optimizer,
    loss='binary_crossentropy',
    metrics=[tf.keras.metrics.AUC(name='auc')]
)

print("\nModel architecture:")
model.summary()

print(f"\nTotal parameters: {model.count_params():,}")

In [ ]:
# Train Model
print("Training LSTM model...")

# Calculate class weights to handle imbalance
class_weight = {0: 1, 1: 10}

# Define callbacks
early_stopping = EarlyStopping(
    monitor='val_auc',
    patience=5,
    mode='max',
    restore_best_weights=True,
    verbose=1
)

model_checkpoint = ModelCheckpoint(
    'lstm_v1.h5',
    monitor='val_auc',
    mode='max',
    save_best_only=True,
    verbose=1
)

# Training parameters
EPOCHS = 20
BATCH_SIZE = 32

print(f"\nTraining configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Class weights: {class_weight}")
print(f"  Early stopping patience: 5 epochs")
print(f"\nStarting training...\n")

# Train the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight,
    callbacks=[early_stopping, model_checkpoint],
    verbose=1
)

print("\nTraining complete!")

# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# AUC plot
axes[1].plot(history.history['auc'], label='Train AUC')
axes[1].plot(history.history['val_auc'], label='Val AUC')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC')
axes[1].set_title('Training and Validation AUC')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

print(f"\nBest validation AUC: {max(history.history['val_auc']):.4f}")

In [ ]:
# Evaluate Model
print("Evaluating model on test set...\n")

# Make predictions
y_pred_proba = model.predict(X_test, verbose=0).flatten()
y_pred = (y_pred_proba > 0.5).astype(int)

# Calculate AUC-ROC
test_auc = roc_auc_score(y_test, y_pred_proba)
print(f"{'='*60}")
print(f"TEST SET PERFORMANCE")
print(f"{'='*60}")
print(f"AUC-ROC: {test_auc:.4f}")
print(f"Target:  0.8200")

if test_auc < 0.82:
    print(f"\n⚠️  BELOW TARGET — Increase class weight to 20 and rerun Cell 9")
else:
    print(f"\n✓ TARGET ACHIEVED")

# Plot ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, linewidth=2, label=f'LSTM (AUC = {test_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve - LSTM Failure Prediction', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print(f"\n{'='*60}")
print(f"CONFUSION MATRIX")
print(f"{'='*60}")

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=['Normal', 'Pre-failure'],
            yticklabels=['Normal', 'Pre-failure'])
plt.xlabel('Predicted', fontsize=12)
plt.ylabel('Actual', fontsize=12)
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Classification Report
print(f"\n{'='*60}")
print(f"CLASSIFICATION REPORT")
print(f"{'='*60}")
print(classification_report(y_test, y_pred, 
                          target_names=['Normal', 'Pre-failure'],
                          digits=4))

# Additional metrics
tn, fp, fn, tp = cm.ravel()
print(f"\n{'='*60}")
print(f"DETAILED METRICS")
print(f"{'='*60}")
print(f"True Negatives:  {tn:,}")
print(f"False Positives: {fp:,}")
print(f"False Negatives: {fn:,}")
print(f"True Positives:  {tp:,}")
print(f"\nSensitivity (Recall): {tp/(tp+fn):.4f}")
print(f"Specificity:          {tn/(tn+fp):.4f}")
print(f"Precision:            {tp/(tp+fp):.4f}")
print(f"F1-Score:             {2*tp/(2*tp+fp+fn):.4f}")

In [ ]:
# Save Model
import os

# Create models directory if it doesn't exist
os.makedirs('ml/lstm/models', exist_ok=True)

# Save the model
model_path = 'ml/lstm/models/lstm_v1.h5'
model.save(model_path)

print(f"{'='*60}")
print(f"MODEL SAVED")
print(f"{'='*60}")
print(f"Path: {model_path}")
print(f"AUC-ROC: {test_auc:.4f}")
print(f"\nModel ready for deployment to OTDT API!")

# Save scaler for inference
import joblib
scaler_path = 'ml/lstm/models/scaler.pkl'
joblib.dump(scaler, scaler_path)
print(f"\nScaler saved: {scaler_path}")

# Save feature columns for inference
import json
feature_config = {
    'feature_columns': feature_columns,
    'sequence_length': SEQUENCE_LENGTH,
    'n_features': n_features
}
config_path = 'ml/lstm/models/feature_config.json'
with open(config_path, 'w') as f:
    json.dump(feature_config, f, indent=2)
print(f"Feature config saved: {config_path}")

print(f"\n{'='*60}")
print(f"TRAINING COMPLETE")
print(f"{'='*60}")